# Introduction to ROSACE (Python)

This notebook ports the R `intro_rosace` vignette to Python.  It walks through
the full ROSACE analysis pipeline using synthetic OCT1-like deep mutational
scanning (DMS) data:

1. Generate synthetic counts with known ground-truth effects
2. Create `AssayGrowth` objects for each of 3 replicates
3. Filter, impute, and normalize each replicate
4. Integrate replicates into an `AssaySetGrowth`
5. Score variants with Simple Linear Regression (SLR)
6. Hypothesis-test and label variants (Neg / Neutral / Pos)
7. Rank top loss-of-function (LOF) and gain-of-function (GOF) variants
8. Inspect the ROSACE Stan model input (without running full MCMC)
9. Visualize score distributions


In [ ]:
import warnings
warnings.filterwarnings("ignore")

%matplotlib inline

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from rosace.run_rosette import generate_effect, generate_count
from rosace.assay import AssayGrowth, AssaySetGrowth
from rosace.preprocessing import filter_data, impute_data, normalize_data, integrate_data
from rosace.slr import run_slr
from rosace.utils import output_score
from rosace.run_rosace import gen_rosace_input
from rosace.visualization import score_density


## 1. Generate Synthetic OCT1-like DMS Data

We simulate a DMS experiment on a protein with **40 positions**.  Each position
has 19 amino-acid substitutions plus one synonymous wildtype control named
`_wt_{pos}`.  Effects are drawn from three Gaussian components:

| Class   | Fraction | Mean   | SD  |
|---------|----------|--------|-----|
| Neutral | 60 %     |  0.0   | 0.1 |
| Neg     | 30 %     | −1.5   | 0.3 |
| Pos     | 10 %     |  0.8   | 0.2 |


In [ ]:
rng = np.random.default_rng(42)

amino_acids = list("ACDEFGHIKLMNPQRSTVWY")
wt_pool     = list("ACDEFGHIKLM")        # WT residues drawn from this set
n_pos       = 40

wt_residues = rng.choice(wt_pool, size=n_pos, replace=True)

records = []
for pos_idx, wt in enumerate(wt_residues, start=1):
    muts = [aa for aa in amino_acids if aa != wt]
    for mut in muts:
        roll = rng.random()
        if roll < 0.3:
            effect = float(rng.normal(-1.5, 0.3))
            label  = "Neg"
        elif roll < 0.4:
            effect = float(rng.normal(0.8, 0.2))
            label  = "Pos"
        else:
            effect = float(rng.normal(0.0, 0.1))
            label  = "Neutral"
        records.append({
            "variant": f"p.{wt}{pos_idx}{mut}",
            "pos": pos_idx, "wt": wt, "mut": mut,
            "effect": effect, "label": label,
        })
    # Synonymous wildtype control at each position
    records.append({
        "variant": f"_wt_{pos_idx}",
        "pos": pos_idx, "wt": wt, "mut": wt,
        "effect": 0.0, "label": "synonymous",
    })

effects = pd.DataFrame(records)
print(f"Total variants: {len(effects)}")
print(effects["label"].value_counts())
effects.head()


## 2. Simulate Sequencing Counts (3 Replicates, 4 Timepoints)

`generate_count` simulates cell growth (Poisson) followed by negative-binomial
sequencing sampling for each replicate and timepoint.


In [ ]:
count_config = {
    "rounds":     3,          # 3 growth rounds → t0,t1,t2,t3
    "n_reps":     3,
    "init_count": 150,
    "depth":      600_000,
    "disp":       0.05,
    "disp_start": 0.10,
    "seed":       42,
}

counts_df = generate_count(count_config, effects)
print(f"Shape: {counts_df.shape}")
counts_df.head()


## 3. Create `AssayGrowth` Objects

One `AssayGrowth` per replicate.  Each holds the raw count matrix
(variants × timepoints) and replicate metadata.


In [ ]:
wt_names = [v for v in effects["variant"] if v.startswith("_wt_")]
print(f"Number of WT controls: {len(wt_names)}")

raw_assays = []
for rep in [1, 2, 3]:
    rep_cols = [f"rep_{rep}_t{t}" for t in range(4)]
    counts_arr = counts_df[rep_cols].values.astype(float)
    var_names  = counts_df["variant"].tolist()
    assay = AssayGrowth(
        counts=counts_arr,
        var_names=var_names,
        key="OCT1sim",
        rep=rep,
    )
    raw_assays.append(assay)
    print(f"Replicate {rep}: {assay.counts.shape[0]} variants × {assay.counts.shape[1]} timepoints")


## 4. Filter, Impute, and Normalize

### 4.1 Filtering

Remove variants with:
- More than 50 % missing timepoints (`na_rmax=0.5`)
- Total count < 20 across all timepoints (`min_count=20`)


In [ ]:
filtered_assays = []
for assay in raw_assays:
    fa = filter_data(assay, na_rmax=0.5, min_count=20)
    print(f"Rep {assay.rep}: {assay.counts.shape[0]} → {fa.counts.shape[0]} variants after filtering")
    filtered_assays.append(fa)


### 4.2 Imputation

Replace any remaining `NaN` counts with **zero** (zero-imputation is fast and
conservative; KNN imputation is also available via `method='knn'`).


In [ ]:
imputed_assays = []
for assay in filtered_assays:
    ia = impute_data(assay, method="zero")
    imputed_assays.append(ia)
    nan_count = np.isnan(ia.counts).sum()
    print(f"Rep {assay.rep}: {nan_count} NaN values remaining after imputation")


### 4.3 Normalization

Wildtype (synonymous) normalization:
- Compute `log(count + 0.5) − log(WT_sum + 0.5)` at each timepoint
- Subtract the t = 0 value so all variants start at 0
- Remove WT controls from the output (`wt_rm=True`)


In [ ]:
norm_assays = []
for assay in imputed_assays:
    na = normalize_data(assay, method="wt", wt_var_names=wt_names, wt_rm=True)
    norm_assays.append(na)
    print(f"Rep {assay.rep}: {na.norm_counts.shape[0]} variants in norm_counts (WT removed)")


## 5. Integrate 3 Replicates into `AssaySetGrowth`

`integrate_data` combines two `AssayGrowth` objects via a full outer join.
For three replicates we chain two calls: first combine replicates 1 and 2,
then add replicate 3 using a small helper that merges `AssaySetGrowth` with a
third `AssayGrowth`.


In [ ]:
def integrate_three(assays):
    """Integrate 3 AssayGrowth objects into one AssaySetGrowth."""
    a1, a2, a3 = assays

    # Build per-replicate normalized DataFrames
    def norm_df(a):
        names = a.norm_var_names or a.var_names
        T = a.norm_counts.shape[1]
        return pd.DataFrame(
            a.norm_counts,
            index=names,
            columns=[f"r{a.rep}_t{t}" for t in range(T)],
        )

    def raw_df(a):
        T = a.counts.shape[1]
        return pd.DataFrame(
            a.counts,
            index=a.var_names,
            columns=[f"r{a.rep}_raw_t{t}" for t in range(T)],
        )

    combined = norm_df(a1).join(norm_df(a2), how="outer").join(norm_df(a3), how="outer")
    raw_combined = raw_df(a1).join(raw_df(a2), how="outer").join(raw_df(a3), how="outer")

    return AssaySetGrowth(
        combined_counts=combined.values,
        var_names=list(combined.index),
        reps=[a.rep for a in assays],
        key=assays[0].key,
        raw_counts=raw_combined.values,
        rounds=[a.rounds for a in assays],
    )


assay_set = integrate_three(norm_assays)
print(f"AssaySetGrowth: {assay_set.combined_counts.shape[0]} variants, "
      f"{assay_set.combined_counts.shape[1]} columns (3 reps × 4 timepoints)")
print(f"Replicates: {assay_set.reps},  rounds per rep: {assay_set.rounds}")


## 6. Variant Metadata

Parse position, wildtype residue, and mutation from the HGVS-style variant
names, and assign a mutation type (`missense` or `synonymous`).


In [ ]:
var_meta = effects[["variant", "pos", "wt", "mut", "label"]].copy()
var_meta = var_meta.rename(columns={"pos": "position", "label": "type"})
var_meta["type"] = var_meta["type"].map({
    "Neg": "missense", "Pos": "missense", "Neutral": "missense",
    "synonymous": "synonymous",
})

# Keep only variants present in the integrated assay set
assay_vars = set(assay_set.var_names)
var_meta = var_meta[var_meta["variant"].isin(assay_vars)].reset_index(drop=True)
print(f"Metadata rows: {len(var_meta)}")
var_meta["type"].value_counts()


## 7. Score Variants with Simple Linear Regression (SLR)

SLR is the *naive* scoring method: for each variant it fits an OLS regression
of normalized count vs. timepoint index and uses the **slope** as the
functional score.  This is fast and serves as a baseline before running the
full Bayesian ROSACE model.


In [ ]:
score_obj = run_slr(assay_set)
print(f"Method : {score_obj.method}")
print(f"Type   : {score_obj.type}")
print(f"Assay  : {score_obj.assay_name}")
print(f"Scored variants: {len(score_obj.score)}")
score_obj.score.head()


## 8. Hypothesis Testing

`output_score` computes the **local false sign rate (LFSR)** from each
variant's `mean` and `sd`, then labels variants at a significance threshold
of `sig_test = 0.05`:

| LFSR              | Label   |
|-------------------|---------|
| ≥ 0.05            | Neutral |
| < 0.05, mean > 0  | Pos     |
| < 0.05, mean ≤ 0  | Neg     |


In [ ]:
scores = output_score(score_obj, sig_test=0.05)
print("Label distribution:")
print(scores["label"].value_counts())
scores.head()


## 9. Merge Scores with Variant Metadata


In [ ]:
scores_meta = scores.merge(
    var_meta[["variant", "position", "wt", "mut", "type"]],
    on="variant", how="left",
)
print(f"Merged DataFrame shape: {scores_meta.shape}")
scores_meta.head()


## 10. Top Loss-of-Function and Gain-of-Function Variants


In [ ]:
missense = scores_meta[scores_meta["type"] == "missense"].copy()

cols = ["variant", "position", "wt", "mut", "mean", "sd", "lfsr", "label"]

print("Top 5 Loss-of-Function variants (most negative score):")
display(missense.nsmallest(5, "mean")[cols].reset_index(drop=True))

print("\nTop 5 Gain-of-Function variants (most positive score):")
display(missense.nlargest(5, "mean")[cols].reset_index(drop=True))


## 11. Position-Level Summary

Aggregate per-position mean score and count the fraction of Neg/Pos calls.


In [ ]:
pos_summary = (
    missense.groupby("position")
    .agg(
        mean_score=("mean", "mean"),
        n_variants=("variant", "count"),
        n_neg=("label", lambda s: (s == "Neg").sum()),
        n_pos=("label", lambda s: (s == "Pos").sum()),
    )
    .reset_index()
)
pos_summary["frac_neg"] = pos_summary["n_neg"] / pos_summary["n_variants"]
pos_summary["frac_pos"] = pos_summary["n_pos"] / pos_summary["n_variants"]

print("Top 5 positions by most negative mean score:")
display(pos_summary.nsmallest(5, "mean_score").reset_index(drop=True))


## 12. ROSACE Stan Model Input

`gen_rosace_input` prepares the data dictionary that would be passed to the
Stan MCMC sampler.  We inspect its structure here without running the full
model (which requires `cmdstanpy` and CmdStan to be installed).


In [ ]:
# Build var_info for ROSACE1 (needs variant, pos, wt, mut columns)
var_info = var_meta.rename(columns={"position": "pos"})[["variant", "pos", "wt", "mut"]].copy()
# Keep only the variants present in assay_set
var_info = var_info[var_info["variant"].isin(assay_set.var_names)].reset_index(drop=True)

stan_data = gen_rosace_input(
    assay=assay_set,
    method="ROSACE1",
    var_info=var_info,
)

print("Stan data keys and shapes/values:")
for k, v in stan_data.items():
    if isinstance(v, list):
        print(f"  {k}: list of length {len(v)}, first element = {v[0]!r}")
    else:
        print(f"  {k}: {v!r}")


## 13. Score Distribution Visualisation

`score_density` plots KDE density curves of the functional scores, faceted by
mutation type.


In [ ]:
fig = score_density(
    data=scores_meta.dropna(subset=["type"]),
    type_col="type",
    score_col="mean",
    order=["missense", "synonymous"],
    title="SLR Score Distribution – OCT1 Simulation",
    show=False,   # notebook will display inline
)
plt.tight_layout()
plt.show()


## 14. Hypothesis Testing Summary

Confusion matrix comparing ROSACE SLR labels against the ground-truth
simulation labels.


In [ ]:
# Merge with ground-truth labels from the simulation
gt = effects[["variant", "label"]].rename(columns={"label": "true_label"})
comparison = scores_meta.merge(gt, on="variant", how="inner")

conf = pd.crosstab(
    comparison["true_label"],
    comparison["label"],
    rownames=["True label"],
    colnames=["SLR label"],
)
print("Confusion matrix (rows = ground truth, columns = SLR call):")
display(conf)

# Overall accuracy on clearly non-neutral ground truth variants
non_neutral = comparison[comparison["true_label"].isin(["Neg", "Pos"])]
correct = (non_neutral["true_label"] == non_neutral["label"]).sum()
total   = len(non_neutral)
print(f"\nAccuracy on Neg/Pos ground-truth variants: {correct}/{total} = {correct/total:.1%}")


## Summary

This notebook demonstrated the complete ROSACE Python workflow on synthetic
OCT1-like DMS data:

1. **Data simulation** – synthetic counts with known ground-truth effects
2. **QC** – per-replicate filtering, zero imputation, WT normalization
3. **Integration** – three replicates merged into a single `AssaySetGrowth`
4. **SLR scoring** – fast linear-regression baseline scores
5. **Hypothesis testing** – LFSR-based Neg / Neutral / Pos labels
6. **Interpretation** – top LOF/GOF variants, position-level summaries
7. **ROSACE model input** – Stan data dictionary for full Bayesian inference
8. **Visualisation** – score density plots by mutation type

For the full Bayesian ROSACE analysis replace `run_slr(assay_set)` with
`run_rosace(assay_set, method="ROSACE1", var_info=var_info)` (requires
CmdStan and `cmdstanpy`).
